# M10.3 · CPU'da paralel çekirdekler — integral dizi/görüntü, 2B konvolüsyon, matris çarpımı

Paralel programlama alıştırmalarının CPU yarısı: tek ve çok thread'li integral dizi (iki geçişli blok taraması), integral görüntü (önce satırlar, sonra sütunlar), uint8 girişli/fp32 hesaplı 3×3…21×21 2B konvolüsyon (satır bantları ve kenar "halo"su) ve fp32 matris çarpımı (satır blokları, devrik B, karo/blok). Her birinde iş bölme (geometrik ayrıştırma), bağımlılıkların nasıl kırıldığı ve thread sayısı ile problem boyunun hızlanmaya etkisi. Saf Python'da bantlı/bloklu algoritmalar, NumPy'da vektörel karşılıkları ve ayrılabilir filtre, C++ `std::thread` ile gerçek uygulamalar, sonda PyTorch'un kendi çekirdekleriyle karşılaştırma ve küçük problemlerde thread ek yükü.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m10-3-cpu-parallel.html](https://m-gonul.github.io/pytorch-lab/m10-3-cpu-parallel.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m10_3_cpu_parallel.py
CHECKS = {
 "b": "import random\nimport numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    rng = random.Random(0)\n\n    def scan():\n        ensure(need(ns, \"block_scan\")(list(range(1, 9)), 4) == [1, 3, 6, 10, 15, 21, 28, 36], \"el örneği\")\n        for n, p in ((1000, 7), (13, 4), (5, 8)):\n            a = [rng.randint(-50, 50) for _ in range(n)]\n            ensure(ns[\"block_scan\"](a, p) == list(np.cumsum(a)), f\"n = {n}, p = {p}\")\n        img = [[rng.randint(0, 255) for _ in range(23)] for _ in range(17)]\n        for p in (1, 3, 5):\n            ensure(need(ns, \"integral_image\")(img, p) == np.cumsum(np.cumsum(img, 0), 1).tolist(), f\"integral görüntü p = {p}\")\n    R_.test(\"taramalar\", scan)\n\n    def conv():\n        img = [[rng.randint(0, 255) for _ in range(20)] for _ in range(18)]\n        g = np.exp(-((np.arange(5) - 2) ** 2) / 2.0)\n        k = (np.outer(g, g) / np.outer(g, g).sum()).astype(np.float32).tolist()\n        ref = need(ns, \"conv_bands\")(img, k, 1)\n        for p in (2, 3, 7):\n            ensure(ns[\"conv_bands\"](img, k, p) == ref, f\"bant sayısı sonucu değiştirmemeli (p = {p})\")\n        f = np.pad(np.array(img, float), 2)\n        direct = np.array([[(f[y:y + 5, x:x + 5] * np.array(k)).sum() for x in range(20)] for y in range(18)])\n        ensure(np.abs(np.array(ref) - np.clip(np.floor(direct + 0.5), 0, 255)).max() <= 1, \"doğrudan konvolüsyonla aynı\")\n        A = [[rng.random() for _ in range(9)] for _ in range(11)]\n        B = [[rng.random() for _ in range(7)] for _ in range(9)]\n        close(np.array(need(ns, \"matmul_rows\")(A, B, 4)), np.array(A) @ np.array(B), what=\"matmul\")\n    R_.test(\"konvolüsyon ve matmul\", conv)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"scan_blocks\", \"integral_image\", \"conv2d\", \"conv2d_separable\", \"matmul_tiled\")])\n    rng = np.random.default_rng(0)\n\n    def t():\n        a = rng.integers(-100, 100, 4096)\n        close(need(ns, \"scan_blocks\")(a, 8), np.cumsum(a), what=\"blok taraması\")\n        img = rng.integers(0, 256, (64, 80)).astype(np.uint8)\n        close(need(ns, \"integral_image\")(img), np.cumsum(np.cumsum(img.astype(np.int64), 0), 1), what=\"integral görüntü\")\n        g = np.exp(-((np.arange(7) - 3) ** 2) / 4.0).astype(np.float32)\n        g /= g.sum()\n        K = np.outer(g, g).astype(np.float32)\n        c2 = need(ns, \"conv2d\")(img, K)\n        f = np.pad(img.astype(np.float64), 3)\n        direct = np.array([[(f[y:y + 7, x:x + 7] * K).sum() for x in range(80)] for y in range(64)])\n        ensure(c2.dtype == np.uint8 and np.abs(c2.astype(int) - np.clip(np.floor(direct + 0.5), 0, 255)).max() <= 1, \"2B konvolüsyon\")\n        cs = need(ns, \"conv2d_separable\")(img, g, g)\n        ensure(np.abs(cs.astype(int) - c2.astype(int)).max() <= 1, \"ayrılabilir = 2B (yuvarlama payı)\")\n        A = rng.random((96, 96)).astype(np.float32)\n        B = rng.random((96, 96)).astype(np.float32)\n        close(need(ns, \"matmul_tiled\")(A, B, 16), A @ B, rtol=1e-4, what=\"karolu matmul\")\n        big = rng.integers(0, 256, (256, 256)).astype(np.uint8)\n        g21 = np.exp(-((np.arange(21) - 10) ** 2) / 50.0).astype(np.float32)\n        g21 /= g21.sum()\n        t2 = bench(lambda: ns[\"conv2d\"](big, np.outer(g21, g21).astype(np.float32)), repeat=2, number=1)\n        t1 = bench(lambda: ns[\"conv2d_separable\"](big, g21, g21), repeat=2, number=1)\n        R_.table(\"256×256, 21×21 Gauss\", [\"yöntem\", \"süre\"], [[\"2B (441 çarpım/piksel)\", fmt_time(t2)], [\"ayrılabilir (42 çarpım/piksel)\", fmt_time(t1)]])\n    R_.test(\"vektörel çekirdekler\", t)\n",
 "d": "import os\nimport time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure, fmt_time\n\n\ndef best(f, r=3):\n    b = 1e9\n    for _ in range(r):\n        t0 = time.perf_counter()\n        f()\n        b = min(b, time.perf_counter() - t0)\n    return b\n\n\ndef gauss(K):\n    g = torch.exp(-((torch.arange(K) - K // 2).float() ** 2) / (2 * (K / 5) ** 2))\n    k = torch.outer(g, g)\n    return (k / k.sum()).float()\n\n\ndef ref_conv(img, k):\n    K = k.shape[0]\n    acc = F.conv2d(img.float()[None, None], k[None, None], padding=K // 2)[0, 0]\n    return torch.clamp((acc + 0.5).floor(), 0, 255).to(torch.uint8)\n\n\ndef check(ns, R_):\n    ext = need(ns, \"build\")()\n    P = max(2, min(8, os.cpu_count() or 2))\n    g = torch.Generator().manual_seed(0)\n\n    def corr():\n        for n in (1, 7, 4096, 100003):\n            x = torch.randint(-1000, 1000, (n,), dtype=torch.int32, generator=g)\n            for p in (1, 3, P):\n                ensure(torch.equal(ext.integral_array(x, p), torch.cumsum(x.long(), 0)), f\"integral dizi n = {n}, p = {p}\")\n        img = torch.randint(0, 256, (123, 77), dtype=torch.uint8, generator=g)\n        for p in (1, 2, 5):\n            ensure(torch.equal(ext.integral_image(img, p), img.long().cumsum(0).cumsum(1)), f\"integral görüntü p = {p}\")\n        for K in (3, 7, 11, 21):\n            k = gauss(K)\n            out = ext.conv2d(img, k, 3)\n            ref = ref_conv(img, k)\n            d = (out.int() - ref.int()).abs()\n            ensure(int(d.max()) <= 1 and float((d == 0).double().mean()) > 0.99, f\"konvolüsyon K = {K}\")\n            ensure(torch.equal(out, ext.conv2d(img, k, 1)), \"thread sayısı sonucu değiştirmemeli\")\n        A = torch.randn(70, 90, generator=g)\n        B = torch.randn(90, 50, generator=g)\n        for mode in (\"ijk\", \"ikj\", \"bt\", \"tiled\"):\n            close(ext.matmul(A, B, 3, mode).numpy(), (A @ B).numpy(), atol=1e-4, what=f\"matmul {mode}\")\n    R_.test(\"doğruluk\", corr)\n\n    def timing():\n        x = torch.randint(-1000, 1000, (4_000_000,), dtype=torch.int32, generator=g)\n        img = torch.randint(0, 256, (2048, 2048), dtype=torch.uint8, generator=g)\n        im1 = img[:1024, :1024].contiguous()\n        A = torch.randn(512, 512, generator=g)\n        B = torch.randn(512, 512, generator=g)\n        jobs = [(\"integral dizi 4 M\", lambda p: ext.integral_array(x, p)), (\"integral görüntü 2048²\", lambda p: ext.integral_image(img, p)),\n                (\"konv. 1024², 7×7\", lambda p: ext.conv2d(im1, gauss(7), p)), (\"konv. 1024², 21×21\", lambda p: ext.conv2d(im1, gauss(21), p)),\n                (\"matmul 512 ijk\", lambda p: ext.matmul(A, B, p, \"ijk\")), (\"matmul 512 ikj\", lambda p: ext.matmul(A, B, p, \"ikj\")),\n                (\"matmul 512 bt\", lambda p: ext.matmul(A, B, p, \"bt\")), (\"matmul 512 tiled\", lambda p: ext.matmul(A, B, p, \"tiled\"))]\n        rows = []\n        for name, f in jobs:\n            t1 = best(lambda: f(1), 2)\n            tp = best(lambda: f(P), 3)\n            rows.append([name, fmt_time(t1), fmt_time(tp), f\"{t1 / tp:.2f}×\"])\n        R_.table(f\"C++ std::thread, 1 vs {P} thread\", [\"çekirdek\", \"1 thread\", f\"{P} thread\", \"hızlanma\"], rows,\n                 note=\"Bellek-bağımlı çekirdekler (integral dizi/görüntü) bant genişliğinde doyar; hesap-bağımlı olanlar (büyük filtreli konvolüsyon, matmul) çekirdek sayısıyla ölçeklenir. Matmul'da döngü sırası thread sayısından daha büyük fark yaratabilir.\")\n    R_.test(\"ölçümler\", timing)\n",
 "e": "import os\nimport hashlib\nimport torch\nfrom torch.utils.cpp_extension import load_inline\nfrom labcheck import need, ensure, fmt_time\n\nSRC = r\"\"\"\n#include <torch/extension.h>\n#include <thread>\n#include <vector>\n#include <algorithm>\n\ntemplate <class F>\nstatic void parallel(int64_t nthreads, F f) {\n    if (nthreads <= 1) { f(0); return; }\n    std::vector<std::thread> ts;\n    for (int64_t t = 0; t < nthreads; ++t) ts.emplace_back(f, t);\n    for (auto& th : ts) th.join();\n}\n\ntorch::Tensor integral_array(torch::Tensor x, int64_t nthreads) {\n    TORCH_CHECK(x.scalar_type() == torch::kInt32 && x.is_contiguous(), \"int32, bitişik\");\n    const int64_t n = x.numel();\n    const int32_t* a = x.data_ptr<int32_t>();\n    auto out = torch::empty({n}, torch::kInt64);\n    int64_t* s = out.data_ptr<int64_t>();\n    std::vector<int64_t> tot(nthreads, 0);\n    parallel(nthreads, [&](int64_t t) {\n        int64_t acc = 0;\n        for (int64_t i = n * t / nthreads; i < n * (t + 1) / nthreads; ++i) { acc += a[i]; s[i] = acc; }\n        tot[t] = acc;\n    });\n    std::vector<int64_t> off(nthreads, 0);\n    for (int64_t t = 1; t < nthreads; ++t) off[t] = off[t - 1] + tot[t - 1];\n    parallel(nthreads, [&](int64_t t) {\n        if (t == 0) return;\n        for (int64_t i = n * t / nthreads; i < n * (t + 1) / nthreads; ++i) s[i] += off[t];\n    });\n    return out;\n}\n\ntorch::Tensor integral_image(torch::Tensor img, int64_t nthreads) {\n    TORCH_CHECK(img.scalar_type() == torch::kUInt8 && img.dim() == 2 && img.is_contiguous(), \"uint8 (H, W)\");\n    const int64_t H = img.size(0), W = img.size(1);\n    const uint8_t* f = img.data_ptr<uint8_t>();\n    auto out = torch::empty({H, W}, torch::kInt64);\n    int64_t* I = out.data_ptr<int64_t>();\n    parallel(nthreads, [&](int64_t t) {\n        for (int64_t y = H * t / nthreads; y < H * (t + 1) / nthreads; ++y) {\n            int64_t acc = 0;\n            for (int64_t x = 0; x < W; ++x) { acc += f[y * W + x]; I[y * W + x] = acc; }\n        }\n    });\n    parallel(nthreads, [&](int64_t t) {\n        const int64_t x0 = W * t / nthreads, x1 = W * (t + 1) / nthreads;\n        for (int64_t y = 1; y < H; ++y)\n            for (int64_t x = x0; x < x1; ++x) I[y * W + x] += I[(y - 1) * W + x];\n    });\n    return out;\n}\n\ntorch::Tensor conv2d(torch::Tensor img, torch::Tensor k, int64_t nthreads) {\n    TORCH_CHECK(img.scalar_type() == torch::kUInt8 && img.dim() == 2 && img.is_contiguous(), \"uint8 (H, W)\");\n    TORCH_CHECK(k.scalar_type() == torch::kFloat32 && k.dim() == 2 && k.size(0) == k.size(1) && k.size(0) % 2 == 1, \"fp32 tek kenarlı kare filtre\");\n    k = k.contiguous();\n    const int64_t H = img.size(0), W = img.size(1), K = k.size(0), r = K / 2;\n    const uint8_t* f = img.data_ptr<uint8_t>();\n    const float* w = k.data_ptr<float>();\n    auto out = torch::empty({H, W}, torch::kUInt8);\n    uint8_t* o = out.data_ptr<uint8_t>();\n    parallel(nthreads, [&](int64_t t) {\n        for (int64_t y = H * t / nthreads; y < H * (t + 1) / nthreads; ++y) {\n            for (int64_t x = 0; x < W; ++x) {\n                float acc = 0.f;\n                for (int64_t u = 0; u < K; ++u) {\n                    const int64_t yy = y + u - r;\n                    if (yy < 0 || yy >= H) continue;\n                    const uint8_t* row = f + yy * W;\n                    const float* wr = w + u * K;\n                    for (int64_t v = 0; v < K; ++v) {\n                        const int64_t xx = x + v - r;\n                        if (xx >= 0 && xx < W) acc += wr[v] * row[xx];\n                    }\n                }\n                int q = (int)(acc + 0.5f);\n                o[y * W + x] = (uint8_t)std::min(255, std::max(0, q));\n            }\n        }\n    });\n    return out;\n}\n\ntorch::Tensor matmul(torch::Tensor A, torch::Tensor B, int64_t nthreads, std::string mode) {\n    TORCH_CHECK(A.scalar_type() == torch::kFloat32 && B.scalar_type() == torch::kFloat32 && A.size(1) == B.size(0), \"fp32, uyumlu boyut\");\n    A = A.contiguous();\n    B = B.contiguous();\n    const int64_t n = A.size(0), m = A.size(1), q = B.size(1);\n    auto C = torch::zeros({n, q}, torch::kFloat32);\n    const float* a = A.data_ptr<float>();\n    const float* b = B.data_ptr<float>();\n    float* c = C.data_ptr<float>();\n    torch::Tensor Bt;\n    const float* bt = nullptr;\n    if (mode == \"bt\") { Bt = B.t().contiguous(); bt = Bt.data_ptr<float>(); }\n    const int64_t T = 64;\n    parallel(nthreads, [&](int64_t t) {\n        const int64_t i0 = n * t / nthreads, i1 = n * (t + 1) / nthreads;\n        if (mode == \"ijk\") {\n            for (int64_t i = i0; i < i1; ++i)\n                for (int64_t j = 0; j < q; ++j) {\n                    float s = 0.f;\n                    for (int64_t kk = 0; kk < m; ++kk) s += a[i * m + kk] * b[kk * q + j];\n                    c[i * q + j] = s;\n                }\n        } else if (mode == \"bt\") {\n            for (int64_t i = i0; i < i1; ++i)\n                for (int64_t j = 0; j < q; ++j) {\n                    float s = 0.f;\n                    for (int64_t kk = 0; kk < m; ++kk) s += a[i * m + kk] * bt[j * m + kk];\n                    c[i * q + j] = s;\n                }\n        } else if (mode == \"ikj\") {\n            for (int64_t i = i0; i < i1; ++i)\n                for (int64_t kk = 0; kk < m; ++kk) {\n                    const float av = a[i * m + kk];\n                    for (int64_t j = 0; j < q; ++j) c[i * q + j] += av * b[kk * q + j];\n                }\n        } else {\n            for (int64_t ii = i0; ii < i1; ii += T)\n                for (int64_t kk0 = 0; kk0 < m; kk0 += T)\n                    for (int64_t jj = 0; jj < q; jj += T)\n                        for (int64_t i = ii; i < std::min(ii + T, i1); ++i)\n                            for (int64_t kk = kk0; kk < std::min(kk0 + T, m); ++kk) {\n                                const float av = a[i * m + kk];\n                                for (int64_t j = jj; j < std::min(jj + T, q); ++j) c[i * q + j] += av * b[kk * q + j];\n                            }\n        }\n    });\n    return C;\n}\n\"\"\"\n\n\ndef check(ns, R_):\n    P = max(2, min(8, os.cpu_count() or 2))\n    ext = load_inline(name=\"ptl_m103e_\" + hashlib.md5(SRC.encode()).hexdigest()[:8], cpp_sources=SRC,\n                      functions=[\"integral_array\", \"integral_image\", \"conv2d\", \"matmul\"], extra_cflags=[\"-O3\"], verbose=False)\n\n    def t():\n        res = need(ns, \"compare\")(ext, P)\n        ensure(isinstance(res, dict) and len(res) == 4, \"dört çekirdek\")\n        rows = [[k, fmt_time(a), fmt_time(b), fmt_time(c), f\"{a / b:.2f}×\", f\"{b / c:.2f}\"] for k, (a, b, c) in res.items()]\n        R_.table(f\"Kendi C++ çekirdeklerimiz ({P} thread) ve PyTorch\", [\"çekirdek\", \"C++ 1 thread\", f\"C++ {P} thread\", \"PyTorch\", \"thread hızlanması\", \"bizim / PyTorch\"], rows,\n                 note=\"PyTorch'un cumsum'u da bellek-bağımlıdır (yakın sonuç); conv2d ve matmul ise vektörleştirilmiş (SIMD), paketlenmiş, çok thread'li kütüphane çekirdekleri (oneDNN/MKL) — aradaki fark Goto–van de Geijn ödevinin konusu.\")\n        sw = need(ns, \"size_sweep\")(ext, [64, 128, 256, 512, 1024, 2048], P)\n        ensure(len(sw) == 6 and sw[0][3] < sw[-1][3], \"küçük görüntüde hızlanma büyükten az olmalı\")\n        R_.table(\"Integral görüntü: boyuta göre hızlanma\", [\"kenar\", \"1 thread\", f\"{P} thread\", \"hızlanma\"], [[s_, fmt_time(a), fmt_time(b), f\"{c:.2f}×\"] for s_, a, b, c in sw],\n                 note=\"Her çağrıda iki kez thread oluşturup birleştiriyoruz (~10–50 µs); 64×64 görüntünün kendisi birkaç µs. Thread havuzu (M10.2 e) bu sabit maliyeti azaltır.\")\n    R_.test(\"karşılaştırma\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        ensure(need(ns, \"separable_error\")(21, 3.0) < 1e-15, \"Gauss ayrılabilir\")\n        x = np.arange(15) - 7\n        X, Y = np.meshgrid(x, x)\n        s = 2.5\n        log = (X ** 2 + Y ** 2 - 2 * s ** 2) * np.exp(-(X ** 2 + Y ** 2) / (2 * s ** 2))\n        disk = ((X ** 2 + Y ** 2) <= 36).astype(float)\n        rnd = np.random.default_rng(0).normal(size=(15, 15))\n        rows = []\n        for name, k in [(\"Gauss σ = 2.5\", np.exp(-(X ** 2 + Y ** 2) / (2 * s ** 2))), (\"LoG\", log), (\"disk r = 6\", disk), (\"rastgele\", rnd)]:\n            errs = [need(ns, \"svd_rank_approx\")(k, r)[1] for r in (1, 2, 3, 5)]\n            rows.append([name] + [f\"{e:.1e}\" for e in errs])\n        ensure(float(rows[1][2]) < 1e-12, \"LoG = iki ayrılabilir terimin toplamı (rank 2)\")\n        c = need(ns, \"cost\")(15, 2)\n        ensure(c == (60, 225), \"maliyet\")\n        R_.table(\"15×15 filtrelerde rank-r ayrılabilir yaklaşımın göreli hatası\", [\"filtre\", \"r = 1\", \"r = 2\", \"r = 3\", \"r = 5\"], rows,\n                 note=\"Maliyet r bileşen için 2Kr, tam 2B için K². LoG = (x² − 2σ²)g(x)g(y) + g(x)(y² − ...)g(y): tam olarak rank 2. Disk gibi dairesel simetrik ama Gauss olmayan çekirdekler birkaç bileşenle iyi yaklaşır; rastgele çekirdek tam ranklıdır.\")\n    R_.test(\"ayrılabilirlik\", t)\n",
 "x2": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        n = 32\n        ensure(sum(1 for _ in need(ns, \"trace\")(\"ijk\", 4)) == 4 * 4 * (2 * 4 + 1), \"ijk iz uzunluğu\")\n        ensure(sorted(set(need(ns, \"trace\")(\"tiled\", 8, 4))) == sorted(set(ns[\"trace\"](\"ijk\", 8))), \"aynı adres kümesi\")\n        rows = []\n        res = {}\n        for order, T in ((\"ijk\", 4), (\"ikj\", 4), (\"tiled\", 4), (\"tiled\", 8)):\n            m = need(ns, \"misses\")(order, n, 8, 32, T)\n            res[(order, T)] = m\n            rows.append([order if order != \"tiled\" else f\"karo T = {T}\", m, f\"{m / (3 * n * n / 8):.1f}×\"])\n        ensure(res[(\"ijk\", 4)] > res[(\"ikj\", 4)] > res[(\"tiled\", 8)], \"ijk > ikj > karo\")\n        R_.table(f\"n = {n}, satır 8 kelime, önbellek 32 satır (LRU)\", [\"sıra\", \"kaçırma\", \"zorunlu kaçırmaya oranı\"], rows,\n                 note=\"Zorunlu (soğuk) kaçırma = 3n²/8 satır. ijk B'yi sütun boyunca okur: her erişim yeni satır. ikj ardışık okur ama B'nin tamamı önbelleğe sığmaz. Karo, üç küçük bloğu önbellekte tutup her elemanı T kez yeniden kullanır; T, 3T² ≤ önbellek olacak şekilde seçilir.\")\n    R_.test(\"önbellek simülasyonu\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"box_filter döngüsüz\", lambda: max_loops(src, 0, func=\"box_filter\"))\n\n    def t():\n        img = np.random.default_rng(0).integers(0, 256, (60, 70)).astype(np.uint8)\n        for r in (0, 1, 5, 40):\n            close(need(ns, \"box_filter\")(img, r), need(ns, \"box_direct\")(img, r), what=f\"r = {r}\")\n        big = np.random.default_rng(1).integers(0, 256, (512, 512)).astype(np.uint8)\n        rows = [[r, fmt_time(bench(lambda: ns[\"box_filter\"](big, r), repeat=3, number=1))] for r in (1, 10, 100)]\n        ensure(True, \"\")\n        R_.table(\"512×512 kutu filtresi (integral görüntüyle)\", [\"yarıçap r\", \"süre\"], rows,\n                 note=\"Süre pencere boyutundan bağımsız: 4 okuma, 3 toplama. Viola–Jones'un Haar öznitelikleri ve hızlı kutu bulanıklığı (ve iterasyonla Gauss yaklaşımı) böyle hesaplanır.\")\n    R_.test(\"kutu filtresi\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m10_3_cpu_parallel as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Paralel programlama alıştırmalarının (BİL 455/533 Ex. 1–2) konusu: aynı çekirdeği tek thread'le ve `std::thread`'lerle yaz, thread sayısını (4, 8, en çok) ve problem boyunu (4096 tamsayı → milyonlar; 100×100 → büyük görüntüler) değiştirerek ölç.

**Integral dizi (önek toplamı):** $S_i=\sum_{j\le i}a_j$ — her eleman öncekine bağlı. **İki geçişli blok taraması:** (1) her thread kendi bloğunu yerel olarak tarar; (2) blok toplamları (p tane) taranır; (3) her thread bloğuna önceki blokların toplamını (ofset) ekler. İş ≈ 2n, yol ≈ 2n/p + p. (Genel paralel tarama M10.6.)

**Integral görüntü:** $I(y,x)=\sum_{y'\le y,x'\le x}f(y',x')$ = satır önek toplamları, sonra sütun önek toplamları. Satır geçişinde satırlar, sütun geçişinde sütun aralıkları bağımsız → iki paralel geçiş (arada bir bariyer: `join`). Uygulama: her boyutta O(1) kutu toplamı (Viola–Jones, M9.16).

**2B konvolüsyon** (alıştırma kuralları): girdi uint8, katsayılar fp32, hesap fp32, çıktı uint8 (yuvarla, [0, 255]'e kırp); filtre 3×3, 7×7, 11×11, 21×21. Geometrik ayrıştırma: çıktı satır bantları; her bant girdinin ±⌊k/2⌋ satırlık **halo**sunu da okur (salt okuma: yarış yok). Maliyet $HWk^2$; ayrılabilir filtrelerde $2HWk$.

**Matris çarpımı:** $2n^3$ FLOP, $3n^2$ veri → aritmetik yoğunluk büyük; ama naif ijk sırası B'nin sütununda yürür (önbellek dostu değil). **ikj** sırası ya da **devrik B** ardışık erişir; **karolama** önbellekte yeniden kullanımı artırır. Paralellik: C'nin satır blokları bağımsız.

Doğruluk kontrolü: "bir lineer cebir kütüphanesiyle karşılaştır" — burada PyTorch.

## 1 · Elle

Bloklar (1,2),(3,4),(5,6),(7,8); toplamlar 3, 7, 11, 15; dışlayıcı tarama 0, 3, 10, 21: 3. blok yerel taraması (5, 11) + 10 → (15, 21) ✓.

$I(1,1)=1+2+3+4$. Konvolüsyon: $512^2\cdot441=115\,605\,504$; ayrılabilir $512^2\cdot42$, oran $441/42=10.5$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İki geçişli blok taraması: [1, 2, …, 8], 4 thread**

- `bs3`: 3. bloğun (5, 6) toplamı
- `off3`: 3. bloğa eklenecek ofset (önceki blokların toplamı)
- `ii`: [[1, 2], [3, 4]] görüntüsünün integral değeri I(1, 1)

**Konvolüsyon, 512 × 512, 21 × 21 filtre**

- `mac`: Çarp-topla sayısı
- `sep`: Ayrılabilir (21 + 21) sürümle oran
- `halo`: Bir satır bandının ek okuduğu (halo) satır sayısı (üst + alt)

**Matmul**

- `flop`: n = 1024 fp32 matmul FLOP sayısı (2n³)

In [ ]:
cevaplar = dict(
    bs3=None,
    off3=None,
    ii=None,
    mac=None,
    sep=None,
    halo=None,
    flop=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"bs3":[11,0],"off3":[10,0],"ii":[10,0],"mac":[115605504,0],"sep":[10.5,0],"halo":[20,0],"flop":[2147483648,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; "thread"ler sıralı döngülerle taklit edilir (önemli olan iş bölümü ve bağımlılıklar).

- `block_scan(a, p)` — iki geçişli kapsayıcı tarama; `p` blok.
- `integral_image(img, p)` — liste listesi; satır geçişi `p` satır grubuna, sütun geçişi `p` sütun grubuna bölünmüş.
- `conv_bands(img, k, p)` — `k` kare katsayı listesi (tek kenar), **sıfır dolgulu "aynı" boyut** korelasyon ($\sum k[u][v]\,f(y+u-r,x+v-r)$, $r=\lfloor K/2\rfloor$), çıktı `min(255, max(0, int(acc + 0.5)))`; satırlar `p` banda bölünür ve her bant yalnız kendi halo'lu girdi dilimini kullanır.
- `matmul_rows(A, B, p)` — C'nin satır blokları, iç çarpım ikj sırasıyla.

In [ ]:
%%writefile stage_b.py
def block_scan(a, p):
    ...


def integral_image(img, p):
    ...


def conv_bands(img, k, p):
    ...


def matmul_rows(A, B, p):
    ...


print(block_scan(list(range(1, 9)), 4))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz.

- `scan_blocks(a, p)` — `len(a)` p'ye bölünür; `reshape(p, -1)` → satır `cumsum` + dışlayıcı blok ofsetleri.
- `integral_image(img)` — `cumsum` iki eksende (int64).
- `conv2d(img, k)` — `sliding_window_view` + `einsum`, fp32, sıfır dolgu, uint8 çıktı (b ile aynı yuvarlama).
- `conv2d_separable(img, kx, ky)` — önce yatay, sonra dikey 1B (fp32).
- `matmul_tiled(A, B, T)` — `n` T'ye bölünür; `A.reshape(n/T, T, n/T, T)` ve `einsum` ile karo karo toplam.

In [ ]:
%%writefile stage_c.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def scan_blocks(a, p):
    ...


def integral_image(img):
    ...


def conv2d(img, k):
    ...


def conv2d_separable(img, kx, ky):
    ...


def matmul_tiled(A, B, T):
    ...


print(scan_blocks(np.arange(1, 9), 4))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

C++ `std::thread`.

- `integral_array(x, nthreads)` — `x` int32 → int64 kapsayıcı tarama (iki geçiş; `nthreads = 1` sıralı).
- `integral_image(img, nthreads)` — uint8 → int64 `(H, W)`; satır geçişi ve sütun geçişi (her thread bir sütun aralığı, satırlarda yürür).
- `conv2d(img, k, nthreads)` — uint8 `(H, W)`, fp32 `(K, K)` → uint8, sıfır dolgu, `std::lround` değil **`(int)(acc + 0.5f)`** ve kırpma (b/c ile aynı).
- `matmul(A, B, nthreads, mode)` — fp32; `mode`: `"ijk"`, `"ikj"`, `"bt"` (B devrik kopyası, iç çarpım), `"tiled"` (64'lük karolar, ikj).

In [ ]:
%%writefile stage_d.py
import hashlib
import torch
from torch.utils.cpp_extension import load_inline

SRC = r"""
#include <torch/extension.h>
#include <thread>
#include <vector>
#include <algorithm>

torch::Tensor integral_array(torch::Tensor x, int64_t nthreads) {
    // TODO
    return torch::zeros({x.numel()}, torch::kInt64);
}

torch::Tensor integral_image(torch::Tensor img, int64_t nthreads) {
    // TODO
    return torch::zeros(img.sizes(), torch::kInt64);
}

torch::Tensor conv2d(torch::Tensor img, torch::Tensor k, int64_t nthreads) {
    // TODO
    return torch::zeros_like(img);
}

torch::Tensor matmul(torch::Tensor A, torch::Tensor B, int64_t nthreads, std::string mode) {
    // TODO
    return torch::zeros({A.size(0), B.size(1)}, torch::kFloat32);
}
"""


def build():
    name = "ptl_m103_" + hashlib.md5(SRC.encode()).hexdigest()[:8]
    return load_inline(name=name, cpp_sources=SRC, functions=["integral_array", "integral_image", "conv2d", "matmul"], extra_cflags=["-O3"], verbose=False)


ext = build()
x = torch.randint(-5, 5, (4096,), dtype=torch.int32)
print((ext.integral_array(x, 4) - torch.cumsum(x.long(), 0)).abs().max())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `compare(ext, threads)` (ext: d aşamasının derlenmiş eklentisi, kontrol verir) → sözlük: her çekirdek için (kendi C++ sürümümüz tek thread, çok thread, PyTorch karşılığı) süreler. PyTorch: `torch.cumsum`, iki `cumsum`, `F.conv2d` (float32), `torch.matmul`.
- `size_sweep(ext, sizes, threads)` — integral görüntüde kenar uzunluğuna göre hızlanma (100×100'den büyük görüntülere).

In [ ]:
%%writefile stage_e.py
import os
import time
import torch
import torch.nn.functional as F


def compare(ext, threads):
    ...


def size_sweep(ext, sizes, threads):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Ayrılabilir filtreler ve SVD ile düşük ranklı yaklaşım

- `separable_error(K, sigma)` — Gauss K×K: dış çarpım ile 2B'nin farkı (≈ 0).
- `svd_rank_approx(kernel, r)` — tekil değer ayrışımıyla r ayrılabilir bileşenin toplamı; Frobenius göreli hatası.
- `cost(K, r)` → $2Kr$ çarp-topla/piksel (r bileşen) vs $K^2$.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def separable_error(K, sigma):
    ...


def svd_rank_approx(kernel, r):
    ...


def cost(K, r):
    ...


print(separable_error(21, 3.0))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Önbellek simülatörü: matmul döngü sıraları ve karolar

**LRU önbellek simülatörü** (saf Python): tam ilişkisel, `n_lines` satır, satır başına `line_words` kelime; adres = (matris, eleman indeksi).

- `trace(order, n, T=4)` — `"ijk"`, `"ikj"`, `"tiled"` (T karolar, ikj) için C = AB'nin okuma/yazma adres dizisi (üreteç).
- `misses(order, n, line_words=8, n_lines=32, T=4)` → kaçırma sayısı.

In [ ]:
%%writefile extra_x2.py
from collections import OrderedDict


def trace(order, n, T=4):
    ...


def misses(order, n, line_words=8, n_lines=32, T=4):
    ...


print(misses("ijk", 16), misses("ikj", 16))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · İntegral görüntüyle her boyutta O(1) kutu filtresi

- `box_filter(img, r)` — integral görüntüden her pikselde $(2r+1)^2$ pencere toplamı (kenarlarda kırpılmış pencere), NumPy, döngüsüz.
- `box_direct(img, r)` — doğrudan (referans; döngü serbest).

In [ ]:
%%writefile extra_x3.py
import numpy as np


def box_filter(img, r):
    ...


def box_direct(img, r):
    ...


print(box_filter(np.arange(16).reshape(4, 4), 1))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Goto & van de Geijn (2008): Yüksek başarımlı matris çarpımının anatomisi

[Goto & van de Geijn — Anatomy of High-Performance Matrix Multiplication (ACM TOMS 2008)](https://doi.org/10.1145/1356052.1356053)

d aşamasındaki matmul'u makalenin katmanlı yapısına dönüştür: B'yi L2'ye sığan $k_c\times n_c$ panellere, A'yı L2'de $m_c\times k_c$ bloklara paketle, en içte $m_r\times n_r$ (ör. 6×16) yazmaç-mikroçekirdeği AVX2 intrinsics (`_mm256_fmadd_ps`) ile yaz; thread'leri $n_c$ ya da $m_c$ döngüsünde paralelleştir. `torch.matmul` (MKL/OpenBLAS) ile GFLOP/s karşılaştır ve makinenin tepe değerine oranını hesapla.


**Adımlar**

1. Önbellek boyutlarını `lscpu` ile öğren; m_c, k_c, n_c'yi makalenin kurallarına göre seç.
2. Paketlemenin süresini ayrıca ölç.

**Kabul ölçütleri**

- Tek thread'de tepe değerin ≥ %60'ı.
- Çok thread'de BLAS'ın ≥ %50'si.

## Makale ödevi: Crow (1984): Toplam alan tabloları

[Crow — Summed-Area Tables for Texture Mapping (SIGGRAPH 1984)](https://doi.org/10.1145/964965.808600)

Integral görüntüyü (toplam alan tablosu) d aşamasındaki paralel C++ ile hesapla; ardından makaledeki gibi değişken boyutlu kutu filtresiyle "doku" bulanıklaştırma (her pikselde farklı pencere) uygula. int32 taşma sınırını (8 bit görüntü için en büyük güvenli boyut) hesapla, fp32 integral görüntünün hassasiyet kaybını büyük görüntülerde ölç ve Viola–Jones'taki kaydırılmış (ortalaması çıkarılmış) çözümle karşılaştır.


**Adımlar**

1. Pencere boyutu konuma göre değişsin (ör. odak bulanıklığı).

**Kabul ölçütleri**

- Değişken pencereli filtre doğrudan toplamla aynı (int64).
- fp32 integral görüntüde 4096² sonrası hata ölçülür ve açıklanır.

## Kaynaklar

- [Crow — Summed-Area Tables for Texture Mapping (SIGGRAPH 1984)](https://doi.org/10.1145/964965.808600) — ©
- [Viola & Jones — Rapid Object Detection using a Boosted Cascade of Simple Features (CVPR 2001)](https://doi.org/10.1109/CVPR.2001.990517) — ©
- [Goto & van de Geijn — Anatomy of High-Performance Matrix Multiplication (ACM TOMS 2008)](https://doi.org/10.1145/1356052.1356053) — ©
- [Barlas — Multicore and GPU Programming (2. baskı), Bölüm 3–4](https://www.elsevier.com/books/multicore-and-gpu-programming/barlas/978-0-12-814120-5) — ©
- [Hwu, Kirk, El Hajj — Programming Massively Parallel Processors (4. baskı), Bölüm 7 ve 11](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — ©